# 🕳️ Reddit RabbitHole RAG v2 — Kaggle runbook

**Before you start (right sidebar → Session options):**
- **Internet: On** (needed for HuggingFace, Arctic Shift, GitHub)
- **Accelerator: GPU T4 x2 or P100** (only the *index* step needs it; ingest/meta/extract run fine on it too)
- **Add-ons → Secrets:** add `HF_TOKEN` (write access) and `GEMINI_API_KEY`. Attach both to this notebook.

**How to run:** go top to bottom. Section A is a 5-minute smoke test on a small slice — do it first. Section B is the full run (long). For the long steps, use **Save Version → Save & Run All (Commit)** so it keeps running after you close the tab.

## 0. Setup

In [ ]:
# ---- EDIT THESE ----------------------------------------------------------------
GITHUB_REPO   = "https://github.com/athxrvxaa/reddit-rabbithole-rag.git"
BRANCH        = "v2"
SOURCE        = "github"   # "github" = clone the branch above | "zip" = use the zip uploaded as a Kaggle Dataset
ARTIFACTS_REPO = "athxrvxaa/rabbithole-artifacts"     # PUBLIC HF dataset repo the Space downloads from
CKPT_REPO      = "athxrvxaa/rabbithole-checkpoints"   # PRIVATE HF dataset repo for the big ingest DB checkpoint
SMOKE_MAX_COMMENTS = 100_000
# --------------------------------------------------------------------------------

import os, glob, subprocess
os.chdir("/kaggle/working")

if SOURCE == "github":
    if not os.path.exists("reddit-rabbithole-rag"):
        subprocess.run(["git", "clone", "-b", BRANCH, GITHUB_REPO, "reddit-rabbithole-rag"], check=True)
else:
    zips = glob.glob("/kaggle/input/**/reddit-rabbithole-rag-v2.zip", recursive=True)
    assert zips, "Add the zip as a Kaggle Dataset (Add Input) first."
    subprocess.run(["unzip", "-q", "-o", zips[0], "-d", "/kaggle/working"], check=True)

os.chdir("/kaggle/working/reddit-rabbithole-rag")
print("repo at", os.getcwd())

In [ ]:
!pip install -q -r requirements-pipeline.txt

In [ ]:
# Load secrets into the environment (subprocesses started with ! inherit them)
from kaggle_secrets import UserSecretsClient
sec = UserSecretsClient()
for name in ("HF_TOKEN", "GEMINI_API_KEY"):
    try:
        os.environ[name] = sec.get_secret(name)
        print(f"{name}: loaded")
    except Exception:
        print(f"{name}: MISSING (add it under Add-ons -> Secrets and attach it to this notebook)")

os.environ["ARTIFACTS_REPO"] = ARTIFACTS_REPO
os.environ["RR_ARTIFACTS_DIR"] = "/kaggle/working/artifacts"

import torch
print("GPU available:", torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else "")

In [ ]:
# Sanity: the code's own tests (takes a few seconds)
!python -m pytest -q

## A. Smoke test (≈5–10 min) — do this first
Streams only the first rows into a **separate** folder, so it never mixes with the full run.
Check the output of `stats`: `op_source` should be almost all `submission`, answers should look sensible, categories not all `other`.

In [ ]:
os.environ["RR_DATA_DIR"] = "/kaggle/working/data_smoke"
!python -m rabbithole ingest --max-comments {SMOKE_MAX_COMMENTS}

In [ ]:
!python -m rabbithole meta --min-comments 5

In [ ]:
!python -m rabbithole extract
!python -m rabbithole stats

In [ ]:
# Eyeball a few pairs: do questions look like real questions and answers like real answers?
import sqlite3, textwrap
con = sqlite3.connect("/kaggle/working/data_smoke/rabbithole.db"); con.row_factory = sqlite3.Row
for r in con.execute("SELECT title, question, answer, op_source FROM pairs ORDER BY RANDOM() LIMIT 5"):
    print("AMA :", r["title"][:90], f"[{r['op_source']}]")
    print("Q   :", textwrap.shorten(r["question"], 200))
    print("A   :", textwrap.shorten(r["answer"], 250), "\n")

**If this looks wrong, stop here** and fix `rabbithole/extract.py` (`ExtractConfig`) before the full run. Common knobs: `min_answer_score`, `max_pairs_per_thread`, `min_thread_comments`.
Common problems: `datasets` script errors → `!pip install "datasets<4"`; Arctic Shift 429s → just re-run the `meta` cell.

## B. Full run
Everything below uses `/kaggle/working/data` (separate from the smoke test).

In [ ]:
os.environ["RR_DATA_DIR"] = "/kaggle/working/data"

# Optional: ONLY if the Kaggle session restarted mid-ingest and you uploaded a checkpoint earlier
RESTORE_CHECKPOINT = False
if RESTORE_CHECKPOINT:
    from huggingface_hub import hf_hub_download
    os.makedirs("/kaggle/working/data", exist_ok=True)
    hf_hub_download(CKPT_REPO, "rabbithole.db", repo_type="dataset", local_dir="/kaggle/working/data", token=os.environ["HF_TOKEN"])
    print("restored checkpoint")

### B0. Pick a sample (the dataset is huge — you only need ~4–5M comments)
The cell below peeks at the source and works out a **thread sample rate**: it keeps a random but fixed ~X% of *whole threads* (so Q&A pairs aren't cut in half), instead of just the first N rows.

- It reads the split size from HF metadata. If that fails, type the IAmA split's row count into `SPLIT_ROWS` yourself (HF dataset page → split dropdown → IAmA).
- Sampling needs a full scan of the split. At ~50k rows/s that's `rows / 50k / 60` minutes — if the split is in the hundreds of millions, set `FAST_FIRST_N = True` to just take the first `TARGET_ROWS` rows instead (faster, but covers only the start of the file).

In [ ]:
TARGET_ROWS   = 5_000_000   # comments to keep (4-5M is plenty)
SPLIT_ROWS    = None        # None = auto-detect; or type the IAmA split's row count
FAST_FIRST_N  = False       # True = skip the full scan, take the first TARGET_ROWS rows

!python -m rabbithole inspect-source --target-rows {TARGET_ROWS}

from rabbithole.config import get_settings
from rabbithole.source_info import split_size, suggest_rate
if SPLIT_ROWS is None:
    SPLIT_ROWS = split_size(get_settings())

if FAST_FIRST_N or not SPLIT_ROWS:
    INGEST_ARGS = f"--max-comments {TARGET_ROWS}"
    print("\nStrategy: first-N slice ->", INGEST_ARGS,
          "" if FAST_FIRST_N else "(split size unknown; set SPLIT_ROWS to use thread sampling instead)")
else:
    RATE = suggest_rate(SPLIT_ROWS, TARGET_ROWS)
    INGEST_ARGS = f"--thread-sample-rate {RATE} --target-rows {int(TARGET_ROWS * 1.3)}"
    print(f"\nStrategy: thread sampling ->", INGEST_ARGS)
    print(f"Expect ~{int(SPLIT_ROWS * RATE):,} kept rows; full scan ~{SPLIT_ROWS / 50_000 / 60:,.0f} min at 50k rows/s")

### B1. Ingest the sample — the long one (resumable; re-run the cell if it stops)
Keep the same settings when resuming (the DB remembers the sample rate and refuses a mismatch).

In [ ]:
!python -m rabbithole ingest {INGEST_ARGS}

In [ ]:
# Checkpoint to a PRIVATE HF repo (Kaggle's disk disappears when the session ends)
!python -m rabbithole upload --file /kaggle/working/data/rabbithole.db --in-repo rabbithole.db --repo {CKPT_REPO} --private

### B2. Thread metadata (real OP + title) → Q&A extraction

In [ ]:
!python -m rabbithole meta

In [ ]:
!python -m rabbithole extract
!python -m rabbithole stats

**Look at the stats before indexing.** Total pairs, number of threads, `op_source`, category spread, biggest threads.
Write the pair/thread counts down — they're your new headline numbers for the README.

### B3. Build the search index (needs GPU for speed)

In [ ]:
!python -m rabbithole index

In [ ]:
# Publish artifacts so the HF Space (and any other machine) can download them
!python -m rabbithole upload --repo {ARTIFACTS_REPO}

## C. Try it
Real embeddings + cross-encoder reranker. The first call downloads the models.

In [ ]:
!python -m rabbithole search "What do people in prison actually do all day?" -k 4

In [ ]:
# Needs GEMINI_API_KEY. If you get a 404, run the next cell and set GEMINI_MODEL.
!python -m rabbithole ask "What is the hardest part of being a nurse?"

In [ ]:
!python -m rabbithole gemini-models

## D. Evaluate (this produces the numbers for your README)

In [ ]:
!python -m rabbithole eval-make -n 200

In [ ]:
!python -m rabbithole eval-retrieval

In [ ]:
!python -m rabbithole calibrate

`calibrate` prints a `suggested_threshold`. Sanity-check it (the seed set is small), then set it as the Space variable `RR_ABSTAIN_THRESHOLD`.

In [ ]:
!python -m rabbithole eval-generation --judge

In [ ]:
# Show the retrieval table to paste into the README
print(open("eval/results/retrieval.md").read())

## E. Wrap up
- Artifacts are on `ARTIFACTS_REPO` → deploy the Space with `deploy/push_space.sh` and set secrets `GEMINI_API_KEY`, `ARTIFACTS_REPO` (+ `RR_ABSTAIN_THRESHOLD`).
- Download `eval/results/*` from the Output tab and paste the table into the README.
- Rotate any key that was ever pasted into a notebook.